# PySpark 5: Spark SQL and the catalog

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-05-spark-sql-and-the-catalog.ipynb)

Half your team writes SQL and half writes Python. Pipelines publish tables that analysts query by name, and a
dropped table must not take the raw data with it. Spark SQL and the catalog are how DataFrames become named,
shared tables.

This lesson runs SQL against DataFrames, writes CTEs and window queries in SQL, passes parameters safely, creates
temporary and permanent views, explores databases and tables through the catalog, and compares managed with
external tables. It ends with table DDL and how Delta Lake and Iceberg extend SQL with `MERGE` and time travel.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 4: reading and writing data](pyspark-04-reading-and-writing-data.ipynb), and basic [SQL](../sql/sql-05-aggregations-and-ctes.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered) and `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints, so a cell can pick out the lines that matter."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

## Query DataFrames with SQL

`df.createOrReplaceTempView("orders")` gives a DataFrame a name in the current session; `spark.sql("...")` runs a
query and returns a DataFrame. Both APIs are parsed into the same logical plan and optimized by the same
Catalyst rules, so choosing between them is about readability, not speed. You can mix them freely: start in SQL,
continue with DataFrame methods, or the reverse.

**Where it matters**
- **Teams with SQL skills:** analysts write the logic, engineers wrap it in tested pipelines.
- **Complex reporting queries** that read more naturally as SQL.

**Watch out for**
- SQL inside strings gets no syntax checking until it runs. Keep queries short or test them.
- Temporary views disappear when the session ends and are visible only in that session.

### Example

Compute revenue per status in SQL and with DataFrames, then compare the physical plans.

In [ ]:
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

by_status_sql = spark.sql("SELECT status, count(*) AS orders, sum(amount) AS revenue FROM orders GROUP BY status")
by_status_df = orders.groupBy("status").agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"))

by_status_sql.orderBy("status").show()
print("same schema:", by_status_sql.schema == by_status_df.schema)
by_status_sql.explain()
by_status_df.explain()

**What to notice:** Both queries return the same rows and schema, and both physical plans have the same shape: a partial
`HashAggregate`, an `Exchange` by `status`, and a final `HashAggregate`. Only the generated column ids differ.

## Write CTEs, CASE and window functions in SQL

Spark SQL supports what you know from other databases: common table expressions (`WITH ... AS`), `CASE WHEN`,
joins, subqueries, set operations and window functions (`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`).
It also has Spark-specific clauses such as `PIVOT`, `LATERAL VIEW explode(...)`, and table-valued `VALUES`.

**Where it matters**
- **Porting warehouse SQL** to Spark with few changes.
- **Readable multi-step logic:** each CTE is one named step.

**Watch out for**
- Dialects differ: Spark has no `QUALIFY` in 4.0, and date functions are named differently from other engines.
- A CTE is not materialized: if two parts of the query use it, it may be computed twice.

### Example

Find each customer's largest order with a CTE and a window function, and label it with `CASE`.

In [ ]:
spark.sql("""
    WITH ranked AS (
        SELECT o.customer_id, c.name, o.order_id, o.amount,
               ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.amount DESC, o.order_id) AS rn
        FROM orders o
        JOIN customers c ON c.customer_id = o.customer_id
    )
    SELECT customer_id, name, order_id, amount,
           CASE WHEN amount >= 1000 THEN 'enterprise' WHEN amount >= 100 THEN 'mid_market' ELSE 'small' END AS tier
    FROM ranked
    WHERE rn = 1
    ORDER BY customer_id
""").show()

**What to notice:** One row per customer who has orders: the window numbered each customer's orders from largest to smallest and the
outer query kept number 1. Customers 7 and 8 do not appear because the inner join found no orders for them.

## Pass parameters safely

Building SQL with f-strings (`f"WHERE status = '{status}'"`) breaks on quotes and opens the door to SQL
injection when values come from users or files. `spark.sql` accepts parameters instead:

- **named parameter markers:** `spark.sql("... WHERE status = :status", args={"status": "paid"})`;
- **DataFrames and identifiers as keyword arguments:** `spark.sql("SELECT * FROM {df}", df=orders)` substitutes
  a DataFrame without registering a view.

**Where it matters**
- **Job parameters** such as a business date or a tenant id.
- **Reusable query functions** that take DataFrames as input.

**Watch out for**
- Parameter markers stand for values, not for table or column names.
- `{df}` substitution is string formatting of identifiers; use `args` for values.

### Example

Run one query with a named parameter, then pass a DataFrame into SQL without a view. Finally, see what an f-string does with a quote.

In [ ]:
spark.sql(
    "SELECT order_id, status, amount FROM orders WHERE status = :status AND amount > :minimum ORDER BY order_id",
    args={"status": "paid", "minimum": 100},
).show()

large = orders.where("amount > 200")
spark.sql("SELECT count(*) AS large_orders, sum(amount) AS total FROM {source}", source=large).show()

status = "paid' OR '1'='1"  # a value that came from outside
injected = spark.sql(f"SELECT count(*) AS n FROM orders WHERE status = '{status}'").first().n
safe = spark.sql("SELECT count(*) AS n FROM orders WHERE status = :status", args={"status": status}).first().n
print("f-string returned", injected, "rows; parameter returned", safe)

**What to notice:** The parameterized query found the three paid orders over 100. The `{source}` placeholder used the `large`
DataFrame directly. The f-string version let the value rewrite the query and returned all 12 orders; the
parameter treated the whole value as one string and matched nothing.

## Temporary, global and permanent views

- A **temporary view** (`createOrReplaceTempView`) lives in one session.
- A **global temporary view** (`createOrReplaceGlobalTempView`) is shared by all sessions of the same
  application, under the `global_temp` database.
- A **permanent view** (`CREATE VIEW` in SQL) is stored in the catalog and survives restarts. It stores the query
  text, not data, so it can only refer to permanent tables.

**Where it matters**
- **Notebook work:** temporary views to name intermediate results.
- **Shared definitions:** a permanent `active_customers` view that every report uses.

**Watch out for**
- A view re-runs its query every time it is read; it is not a cache.
- A permanent view cannot reference a temporary view.

### Example

Create one view of each kind and list them.

In [ ]:
orders.where("status = 'paid'").createOrReplaceTempView("paid_orders")
orders.where("status = 'shipped'").createOrReplaceGlobalTempView("shipped_orders")

spark.sql("CREATE DATABASE IF NOT EXISTS shop")
orders.write.mode("overwrite").saveAsTable("shop.orders")
spark.sql("CREATE OR REPLACE VIEW shop.large_orders AS SELECT * FROM shop.orders WHERE amount >= 200")

print("temporary:", spark.table("paid_orders").count(), "rows")
print("global:   ", spark.table("global_temp.shipped_orders").count(), "rows")
print("permanent:", spark.table("shop.large_orders").count(), "rows")
spark.sql("SHOW VIEWS IN shop").show()

**What to notice:** The global view is read through the `global_temp` database. The permanent view lives in the `shop` database next
to the table it reads, and appears in `SHOW VIEWS`. Temporary views have no database and are listed in every
database's `SHOW VIEWS` output with `isTemporary = true`.

## Explore the catalog

The **catalog** stores databases (also called schemas), tables, views and functions. `spark.catalog` lets you
inspect it from Python: `listDatabases`, `listTables`, `listColumns`, `tableExists`, `currentDatabase` and
`setCurrentDatabase`. SQL has the same information through `SHOW DATABASES`, `SHOW TABLES` and `DESCRIBE`.

On Databricks Unity Catalog and other modern catalogs, names have three levels: `catalog.schema.table`, for
example `main.silver.orders`. The local session catalog used here has two: `schema.table`.

**Where it matters**
- **Jobs that create tables if they do not exist** and check columns before writing.
- **Data discovery** in a notebook: what tables exist, and what columns do they have?

**Watch out for**
- Unqualified names resolve against the current database. Qualify names in jobs so they do not depend on session state.
- The local catalog lives only as long as `spark.sql.warehouse.dir` and the session; production uses a shared metastore.

### Example

List databases and tables, check whether a table exists, and list a table's columns.

In [ ]:
print("current database:", spark.catalog.currentDatabase())
print("databases:", [db.name for db in spark.catalog.listDatabases()])
print("tables in shop:", [(t.name, t.tableType) for t in spark.catalog.listTables("shop")])
print("shop.orders exists:", spark.catalog.tableExists("shop.orders"))
print("shop.returns exists:", spark.catalog.tableExists("shop.returns"))
for column in spark.catalog.listColumns("shop.orders"):
    print(f"  {column.name:<12} {column.dataType}")

**What to notice:** `shop` holds the `orders` table (`MANAGED`) and the `large_orders` view (`VIEW`); the session's temporary views
are listed too, as `TEMPORARY`, because they are visible from every database. `tableExists` lets a job decide
between creating and appending without catching exceptions, and `listColumns` returns the stored schema.

## Managed and external tables

A **managed** table is owned by the catalog: its files live under the warehouse directory, and `DROP TABLE`
deletes the data as well as the definition. An **external** (unmanaged) table points at files in a location
you choose (`.option("path", ...)` or `LOCATION '...'` in SQL); `DROP TABLE` removes only the definition and the
files stay.

**Where it matters**
- **Managed:** tables the platform should fully control, including cleanup and governance.
- **External:** raw or shared data that other engines also read, or data that must survive a dropped table.

**Watch out for**
- Dropping a managed table deletes its data. Check the type with `DESCRIBE EXTENDED` before dropping.
- Two external tables pointing at the same folder both see writes from either.

### Example

Create one managed and one external table, drop both, and check which files survive.

In [ ]:
external_path = os.path.join(WORK_DIR, "external", "customers")
customers.write.mode("overwrite").saveAsTable("shop.customers_managed")
customers.write.mode("overwrite").option("path", external_path).saveAsTable("shop.customers_external")

for table in ["shop.customers_managed", "shop.customers_external"]:
    kind = spark.sql(f"DESCRIBE EXTENDED {table}").where("col_name = 'Type'").first().data_type
    print(f"{table:<26} {kind}")

managed_path = os.path.join(WORK_DIR, "warehouse", "shop.db", "customers_managed")
spark.sql("DROP TABLE shop.customers_managed")
spark.sql("DROP TABLE shop.customers_external")
print("managed files after DROP: ", os.path.exists(managed_path))
print("external files after DROP:", os.path.exists(external_path))
print("external data still readable:", spark.read.parquet(external_path).count(), "rows")

**What to notice:** `DESCRIBE EXTENDED` reports `MANAGED` and `EXTERNAL`. After dropping both, the managed table's folder under the
warehouse is gone, while the external files remain and can still be read or attached to a new table.

## Create and change tables with SQL

The usual table statements work in Spark SQL:

- `CREATE TABLE ... USING parquet PARTITIONED BY (col)` and `CREATE TABLE ... AS SELECT` (CTAS);
- `INSERT INTO` to append and `INSERT OVERWRITE` to replace (with `PARTITION (...)` for one partition);
- `DESCRIBE`, `SHOW PARTITIONS`, `ALTER TABLE ... ADD COLUMNS`.

Plain Parquet tables do not support `UPDATE`, `DELETE` or `MERGE`; those need a table format (next section).

**Where it matters**
- **SQL-first pipelines** and dbt-style models.
- **Schema changes** made explicitly instead of through `mergeSchema`.

**Watch out for**
- `INSERT INTO` matches columns by position unless you list them.
- `INSERT OVERWRITE` without a partition clause replaces the whole table (unless dynamic partition overwrite is on).

### Example

Create a partitioned table, insert two days, overwrite one of them, and add a column.

In [ ]:
spark.sql("DROP TABLE IF EXISTS shop.daily_revenue")
spark.sql("CREATE TABLE shop.daily_revenue (revenue DECIMAL(22,2), orders BIGINT, order_date DATE) USING parquet PARTITIONED BY (order_date)")
spark.sql("""
    INSERT INTO shop.daily_revenue
    SELECT sum(amount), count(*), to_date(order_ts) FROM orders
    WHERE to_date(order_ts) IN (DATE'2026-01-05', DATE'2026-01-12') GROUP BY to_date(order_ts)
""")
spark.sql("INSERT OVERWRITE shop.daily_revenue PARTITION (order_date = DATE'2026-01-12') SELECT 0, 0")
spark.sql("ALTER TABLE shop.daily_revenue ADD COLUMNS (note STRING)")

spark.sql("SHOW PARTITIONS shop.daily_revenue").show(truncate=False)
spark.table("shop.daily_revenue").orderBy("order_date").show()

**What to notice:** The table has one folder per date. `INSERT OVERWRITE ... PARTITION` replaced only 12 January, leaving 5 January
intact. The added `note` column reads as null for existing rows, because old files simply do not contain it.

## Table formats: Delta Lake and Apache Iceberg

Parquet files cannot be changed in place, and a folder of them has no transactions. **Table formats** add a
transaction log on top of Parquet files, which brings:

- atomic commits: readers see the whole write or none of it;
- `UPDATE`, `DELETE` and `MERGE INTO`;
- **time travel:** query an earlier version (`VERSION AS OF 3`, `TIMESTAMP AS OF '...'`);
- schema enforcement and evolution, and file compaction (`OPTIMIZE`).

```sql
MERGE INTO main.silver.orders AS t
USING staged_orders AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.source_version > t.source_version THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT * FROM main.silver.orders VERSION AS OF 12;   -- the table as it was at version 12
DESCRIBE HISTORY main.silver.orders;                  -- every commit (Delta)
```

Using them from Spark needs the Delta or Iceberg Spark package on the classpath; managed platforms ship it.
[Lesson 10](pyspark-10-batch-pipelines.ipynb) runs a version-aware merge with the `deltalake` Python package, which writes the same
Delta format.

**Where it matters**
- **Silver and gold tables** that receive updates and corrections.
- **Audits and recovery:** reading a table as it was before a bad load.

**Watch out for**
- Old versions are kept only until `VACUUM` removes their files (7 days by default in Delta).
- Mixing writers that do not understand the table format (plain Parquet writes into a Delta folder) corrupts the table's history.

### Example

Read the four steps above as a timeline: version 0 is the first write, each later commit adds a version, and
`MERGE` with a version condition keeps late, stale deliveries from overwriting newer rows. The runnable version of
this example is in lesson 10.

## Discover built-in functions

Spark has hundreds of built-in functions. `SHOW FUNCTIONS LIKE 'pattern'` lists them and `DESCRIBE FUNCTION
EXTENDED name` shows usage and an example. Every SQL function is available in Python as `F.name` (or through
`F.expr` when the Python wrapper does not exist yet).

**Where it matters**
- **Before writing a UDF:** check whether a built-in already does it.
- **Learning the API:** examples straight from the engine you run.

**Watch out for**
- Function availability depends on the Spark version; check against the version you deploy.

### Example

Find the functions whose names start with `array_`, and read the documentation of one.

In [ ]:
names = [row.function for row in spark.sql("SHOW FUNCTIONS LIKE 'array_*'").collect()]
print(len(names), "array_ functions:", ", ".join(names[:8]), "...")
for row in spark.sql("DESCRIBE FUNCTION EXTENDED array_distinct").collect()[:4]:
    print(row.function_desc)

**What to notice:** Spark lists every built-in array function, and `DESCRIBE FUNCTION EXTENDED` prints its signature, a description
and a runnable example. This is often faster than searching the documentation.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Is SQL slower or faster than the DataFrame API in Spark?

**A.** Neither. Both are parsed into the same logical plan and optimized by Catalyst into the same physical plan.

**Q2.** Why pass values with `args` instead of an f-string?

**A.** An f-string breaks on quotes and lets a value rewrite the query (SQL injection). A parameter is always treated as a value.

**Q3.** What is the difference between a temporary view and a permanent view?

**A.** A temporary view lives in one session and can refer to anything. A permanent view is stored in the catalog, survives restarts, and can only refer to permanent tables.

**Q4.** What happens to the data when you drop a managed table, and an external one?

**A.** Dropping a managed table deletes its files. Dropping an external table removes only the definition; the files stay.

**Q5.** Why can't you `MERGE` into a plain Parquet table?

**A.** Parquet files are immutable and a folder of them has no transaction log. Delta Lake and Iceberg add the log that makes updates, deletes and merges atomic.

**Q6.** How do you check whether a table exists from Python?

**A.** `spark.catalog.tableExists("db.table")`.

## Key takeaways

- SQL and DataFrames compile to the same plan; use whichever reads better and mix them freely.
- Pass values with `args`, never by formatting SQL strings.
- Qualify table names in jobs; on Unity Catalog that means `catalog.schema.table`.
- Know whether a table is managed or external before you drop it.
- Updates, merges and time travel need a table format such as Delta Lake or Iceberg.

**Next:** Practise this module in [PySpark drills 5: Spark SQL and the catalog](pyspark-drills-05-spark-sql-and-the-catalog.ipynb), then continue with [PySpark 6: joins](pyspark-06-joins.ipynb).